# NOOTEBOOK DE CREACION DE CAPA BASE PARA ETIQUETADO

## entrnemaineto de la red YOLO
Prepara el dataset: lee tus tiles (RGB + NIR), convierte a imágenes 3-ch (opción: RGB normal o R,G,NIR si querés probar NIR en vez del azul), extrae los polígonos generados y los convierte a bounding boxes en formato YOLO (.txt con class x_center y_center width height normalizados).

Crea la estructura train/images, train/labels, test/images, test/labels listos para entrenar.

Genera data.yaml para YOLOv5/YOLOv8.

Te doy el comando para entrenar con YOLOv5 (ultralytics/yolov5).

Opciones y notas para usar NIR como 4ª banda (más avanzado).

Suposiciones hechas (las adapto a lo que pasaste):

Tus tiles son GeoTIFFs y cada tile tiene bandas en el orden que fijás en la configuración (ej. R,G,B,NIR o similar).

Los shapefiles con polígonos siguen el patrón que mostraste: contienen el nombre del tile (por ejemplo 2020_6_3_0_tile_r00_c06_PERCENTILE_POLYGONS.shp) o al menos incluyen el prefijo que identifica la imagen.

Vas a poner tus imágenes de entrenamiento en dataset_raw/train/ y las de prueba en dataset_raw/test/ (puedes cambiar rutas).

Queremos una sola clase: silo bolsa (clase 0). Si después agregás clases, hay que adaptar.

In [6]:
import sys

In [9]:

#!{sys.executable} -m pip install ultralytics

  Using cached numpy-2.2.6-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-win_amd64.whl.metadata (4.1 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 26.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ------------ --------------------------- 11.8/39.0 MB 56.9 MB/s eta 0:00:01
   ------------------- -------------------- 19.4/39.0 MB 46.2 MB/s eta 0:00:01
   ------------------------------ --------- 30.1/39.0 MB 48.4 MB/s eta 0:00:01
   -------------------------------------- - 38.0/39.0 MB 45.4 MB/s eta 0:00:01
   ---------------------------------------  38.8/39.0 MB 45.6 MB/s eta 0:00:01
   ---------------------------------------- 39.0/39.0 MB 33.3 MB/s eta 0:00:00
Using cached numpy-2.2.6-cp313-cp313-win_amd64.whl (12.6 MB)
   --------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
#!{sys.executable} -m pip install rasterio geopandas shapely numpy pillow scipy scikit-image pyproj tqdm



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
##################################### nueva prueba #####################################

In [21]:
# prepare_dataset_with_crops.py
import os
import glob
import math
import random
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.windows import Window
from PIL import Image
from tqdm import tqdm

# --- CONFIG (ajustá) ---
RAW_TRAIN_DIR = r"D:\Silos\Base de datos\YOLO\train"   # GeoTiffs train
RAW_TEST_DIR  = r"D:\Silos\Base de datos\YOLO\test"    # GeoTiffs val/test
SHAPE_DIR     = r"D:\Silos\Base de datos\procesado_sin_nubes\tiles\indices_INBNV\masks\polygons"
OUT_DIR       = r"D:\Silos\Base de datos\YOLO\processed_yolo_v2"
IMG_SIZE      = 640              # tamaño final de las imágenes guardadas (square)
USE_NIR_AS_B  = True             # True => R,G,NIR (reemplaza azul con NIR)
BAND_ORDER    = {"R":1, "G":2, "B":3, "NIR":4}  # 1-indexed
CLASS_ID      = 0

# Crops (en pixeles del raster original) que vamos a extraer alrededor de cada bbox
CROP_SIZES = [256, 512]   # probar varios tamaños; se reescalan a IMG_SIZE
MAX_CROPS_PER_BOX = 2     # cuántos crops por box (aleatoriedad leve)
NEG_CROPS_PER_IMAGE = 2   # crops negativos por imagen (sin cajas)
MIN_BOX_AREA = 1          # no eliminar cajas por ahora; filtrar luego si se desea
# --- FIN CONFIG ---

# crear estructura
for split in ["train","test"]:
    os.makedirs(os.path.join(OUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, split, "labels"), exist_ok=True)

def scale_band(arr):
    """Normaliza banda ignorando NaNs, devuelve uint8."""
    arr = np.array(arr, dtype=np.float32)
    valid = np.isfinite(arr)
    if not np.any(valid):
        return np.zeros_like(arr, dtype=np.uint8)
    p2 = float(np.nanpercentile(arr, 2))
    p98 = float(np.nanpercentile(arr, 98))
    if p98 == p2:
        pmin = float(np.nanmin(arr[valid])); pmax = float(np.nanmax(arr[valid]))
        if pmax == pmin:
            p98 = pmin + 1.0
            p2 = pmin
        else:
            p2, p98 = pmin, pmax
    arr_clipped = np.clip(arr, p2, p98)
    scaled = ((arr_clipped - p2) / (p98 - p2) * 255.0).round().astype(np.uint8)
    scaled[~valid] = 0
    return scaled

def read_rgb_from_window(src, window):
    """Lee R,G,(B|NIR) del raster dentro de window y devuelve H,W,3 uint8"""
    # read bands as float arrays (boundless True to allow windows at edges)
    kwargs = dict(window=window, boundless=True, fill_value=np.nan)
    r = src.read(BAND_ORDER["R"], **kwargs).astype(np.float32)
    g = src.read(BAND_ORDER["G"], **kwargs).astype(np.float32)
    if USE_NIR_AS_B:
        b = src.read(BAND_ORDER["NIR"], **kwargs).astype(np.float32)
    else:
        b = src.read(BAND_ORDER["B"], **kwargs).astype(np.float32)
    r8, g8, b8 = scale_band(r), scale_band(g), scale_band(b)
    img = np.dstack([r8, g8, b8])
    return img

def full_image_to_rgb(src_path):
    """Lee todo el raster y devuelve H,W,3 uint8; devuelve None si todo NaN"""
    with rasterio.open(src_path) as src:
        r = src.read(BAND_ORDER["R"]).astype(np.float32)
        g = src.read(BAND_ORDER["G"]).astype(np.float32)
        if USE_NIR_AS_B:
            b = src.read(BAND_ORDER["NIR"]).astype(np.float32)
        else:
            b = src.read(BAND_ORDER["B"]).astype(np.float32)
    if not (np.isfinite(r).any() or np.isfinite(g).any() or np.isfinite(b).any()):
        return None
    return np.dstack([scale_band(r), scale_band(g), scale_band(b)])

def find_shapefile_for_image(img_path, shp_dir):
    base = os.path.splitext(os.path.basename(img_path))[0]
    candidates = glob.glob(os.path.join(shp_dir, f"{base}*POLYGONS.*")) + glob.glob(os.path.join(shp_dir, f"*{base}*POLYGONS.*"))
    candidates = [c for c in candidates if c.lower().endswith(".shp")]
    if candidates:
        return candidates[0]
    for c in glob.glob(os.path.join(shp_dir, "*.shp")):
        if base in os.path.basename(c):
            return c
    return None

def bboxes_from_shapefile_on_raster(img_path, shp_path):
    if shp_path is None:
        return []
    try:
        gdf = gpd.read_file(shp_path)
    except Exception as e:
        print("Error leyendo:", shp_path, e)
        return []
    with rasterio.open(img_path) as src:
        inv = ~src.transform
        w, h = src.width, src.height
        crs_r = src.crs
    if gdf.crs is not None and crs_r is not None and gdf.crs != crs_r:
        gdf = gdf.to_crs(crs_r)
    bboxes = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        minx, miny, maxx, maxy = geom.bounds
        col_min, row_min = inv * (minx, maxy)
        col_max, row_max = inv * (maxx, miny)
        col_min = max(0, min(col_min, w-1)); col_max = max(0, min(col_max, w-1))
        row_min = max(0, min(row_min, h-1)); row_max = max(0, min(row_max, h-1))
        if (col_max - col_min) < 1 or (row_max - row_min) < 1:
            continue
        bboxes.append((col_min, row_min, col_max, row_max))
    return bboxes

def save_jpg(img_np, path, size=IMG_SIZE):
    im = Image.fromarray(img_np)
    im = im.convert("RGB")
    im = im.resize((size, size), Image.BILINEAR)
    im.save(path, quality=95)

def write_yolo_label_from_boxes(boxes, out_txt_path, orig_w, orig_h, crop_origin=(0,0), crop_w=None, crop_h=None, target_w=IMG_SIZE, target_h=IMG_SIZE):
    """
    boxes = list of (xmin, ymin, xmax, ymax) in original raster pixel coords
    crop_origin = (col0, row0) top-left of crop in original coords
    crop_w/h are dims of crop in original pixels; if None use orig_w/h
    """
    if crop_w is None: crop_w = orig_w
    if crop_h is None: crop_h = orig_h

    lines = []
    cx0, cy0 = crop_origin
    sx = float(target_w) / float(crop_w)
    sy = float(target_h) / float(crop_h)

    for (xmin, ymin, xmax, ymax) in boxes:
        # shift to crop coords
        xmin_c = xmin - cx0
        ymin_c = ymin - cy0
        xmax_c = xmax - cx0
        ymax_c = ymax - cy0
        # clip to crop
        xmin_c = max(0, min(xmin_c, crop_w))
        xmax_c = max(0, min(xmax_c, crop_w))
        ymin_c = max(0, min(ymin_c, crop_h))
        ymax_c = max(0, min(ymax_c, crop_h))
        if (xmax_c - xmin_c) <= 0 or (ymax_c - ymin_c) <= 0:
            continue
        # scale to target and normalize
        xcenter = (xmin_c + xmax_c) / 2.0 * sx / target_w
        ycenter = (ymin_c + ymax_c) / 2.0 * sy / target_h
        w_rel = (xmax_c - xmin_c) * sx / target_w
        h_rel = (ymax_c - ymin_c) * sy / target_h
        lines.append(f"{CLASS_ID} {xcenter:.6f} {ycenter:.6f} {w_rel:.6f} {h_rel:.6f}")

    with open(out_txt_path, "w") as fh:
        fh.write("\n".join(lines))

def generate_negative_crop_windows(img_w, img_h, existing_boxes, crop_size, n_samples=1):
    """
    Genera ventanas aleatorias que no contengan centros de boxes.
    existing_boxes: list of boxes in original pixel coords
    """
    windows = []
    tries = 0
    while len(windows) < n_samples and tries < 200:
        tries += 1
        col0 = random.randint(0, max(0, img_w - crop_size))
        row0 = random.randint(0, max(0, img_h - crop_size))
        ok = True
        for (xmin,ymin,xmax,ymax) in existing_boxes:
            cx = (xmin + xmax) / 2.0
            cy = (ymin + ymax) / 2.0
            if (col0 <= cx <= col0 + crop_size) and (row0 <= cy <= row0 + crop_size):
                ok = False
                break
        if ok:
            windows.append((col0, row0))
    return windows

def process_split(raw_dir, out_images_dir, out_labels_dir, shp_dir, split_name="train"):
    tifs = sorted(glob.glob(os.path.join(raw_dir, "*.tif")))
    for tif in tqdm(tifs, desc=f"{split_name}"):
        base = os.path.splitext(os.path.basename(tif))[0]
        try:
            shp = find_shapefile_for_image(tif, shp_dir)
            boxes = bboxes_from_shapefile_on_raster(tif, shp)  # in original pixel coords
            with rasterio.open(tif) as src:
                orig_w, orig_h = src.width, src.height

                # 1) Full image sample (context)
                full_img = read_rgb_from_window(src, Window(0,0, orig_w, orig_h))
                if full_img is None:
                    continue
                out_img_full = os.path.join(out_images_dir, f"{base}.jpg")
                out_lbl_full = os.path.join(out_labels_dir, f"{base}.txt")
                save_jpg(full_img, out_img_full, size=IMG_SIZE)
                write_yolo_label_from_boxes(boxes, out_lbl_full, orig_w, orig_h, crop_origin=(0,0), crop_w=orig_w, crop_h=orig_h)

                # 2) Crops centered on each bbox
                for box in boxes:
                    xmin,ymin,xmax,ymax = box
                    cx = int((xmin + xmax) / 2.0)
                    cy = int((ymin + ymax) / 2.0)
                    for crop_size in CROP_SIZES:
                        for i in range(MAX_CROPS_PER_BOX):
                            # jitter center a little para diversidad
                            jitter_x = int(random.uniform(-0.15,0.15) * crop_size)
                            jitter_y = int(random.uniform(-0.15,0.15) * crop_size)
                            cx_j = cx + jitter_x
                            cy_j = cy + jitter_y
                            col0 = int(max(0, min(orig_w - crop_size, cx_j - crop_size//2)))
                            row0 = int(max(0, min(orig_h - crop_size, cy_j - crop_size//2)))
                            win = Window(col0, row0, crop_size, crop_size)
                            img_crop = read_rgb_from_window(src, win)
                            if img_crop is None:
                                continue
                            crop_name = f"{base}_C_{crop_size}_{i}"
                            out_img_crop = os.path.join(out_images_dir, f"{crop_name}.jpg")
                            out_lbl_crop = os.path.join(out_labels_dir, f"{crop_name}.txt")
                            save_jpg(img_crop, out_img_crop, size=IMG_SIZE)
                            # select boxes that intersect this crop (shifted)
                            boxes_in_crop = []
                            for b in boxes:
                                # check overlap with window
                                if not (b[2] < col0 or b[0] > col0+crop_size or b[3] < row0 or b[1] > row0+crop_size):
                                    boxes_in_crop.append(b)
                            write_yolo_label_from_boxes(boxes_in_crop, out_lbl_crop, orig_w, orig_h, crop_origin=(col0,row0), crop_w=crop_size, crop_h=crop_size)

                # 3) Negative random crops (sin cajas)
                neg_windows = generate_negative_crop_windows(orig_w, orig_h, boxes, max(CROP_SIZES), n_samples=NEG_CROPS_PER_IMAGE)
                for idx,(col0,row0) in enumerate(neg_windows):
                    win = Window(col0, row0, max(CROP_SIZES), max(CROP_SIZES))
                    img_neg = read_rgb_from_window(src, win)
                    if img_neg is None:
                        continue
                    neg_name = f"{base}_NEG_{idx}"
                    out_img_neg = os.path.join(out_images_dir, f"{neg_name}.jpg")
                    out_lbl_neg = os.path.join(out_labels_dir, f"{neg_name}.txt")
                    save_jpg(img_neg, out_img_neg, size=IMG_SIZE)
                    # write empty label file
                    open(out_lbl_neg, "w").close()

        except Exception as e:
            print("ERROR procesando:", tif, e)

if __name__ == "__main__":
    process_split(RAW_TRAIN_DIR, os.path.join(OUT_DIR,"train","images"), os.path.join(OUT_DIR,"train","labels"), SHAPE_DIR, split_name="train")
    process_split(RAW_TEST_DIR,  os.path.join(OUT_DIR,"test","images"),  os.path.join(OUT_DIR,"test","labels"),  SHAPE_DIR, split_name="test")
    print("Dataset preparado en:", OUT_DIR)


train:   0%|          | 0/10 [00:00<?, ?it/s]C:\Users\m\AppData\Local\Temp\ipykernel_14432\876953152.py:51: RuntimeWarning: invalid value encountered in cast
  scaled = ((arr_clipped - p2) / (p98 - p2) * 255.0).round().astype(np.uint8)
train:  10%|█         | 1/10 [00:06<01:00,  6.74s/it]C:\Users\m\AppData\Local\Temp\ipykernel_14432\876953152.py:51: RuntimeWarning: invalid value encountered in cast
  scaled = ((arr_clipped - p2) / (p98 - p2) * 255.0).round().astype(np.uint8)
train:  20%|██        | 2/10 [00:16<01:09,  8.63s/it]C:\Users\m\AppData\Local\Temp\ipykernel_14432\876953152.py:51: RuntimeWarning: invalid value encountered in cast
  scaled = ((arr_clipped - p2) / (p98 - p2) * 255.0).round().astype(np.uint8)
train:  30%|███       | 3/10 [00:40<01:47, 15.39s/it]C:\Users\m\AppData\Local\Temp\ipykernel_14432\876953152.py:51: RuntimeWarning: invalid value encountered in cast
  scaled = ((arr_clipped - p2) / (p98 - p2) * 255.0).round().astype(np.uint8)
train:  40%|████      | 4/10 [01

Dataset preparado en: D:\Silos\Base de datos\YOLO\processed_yolo_v2


In [22]:
# train_yolo_v2.py
from ultralytics import YOLO

# Ajusta ruta al data.yaml (usa la carpeta OUT_DIR del script anterior)
DATA_YAML = r"D:\Silos\Base de datos\YOLO\processed_yolo_v2\data.yaml"
WEIGHTS = "yolov8s.pt"   # o yolov8m.pt si tenés más recursos
IMG_SIZE = 640           # debe coincidir con IMG_SIZE en prepare script
EPOCHS = 80
BATCH = 8                # ajustá según VRAM; en CPU pon 2-4
NAME = "silo_exp_aug_crops"

model = YOLO(WEIGHTS)
model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    name=NAME,
    augment=True,         # habilita augmentaciones
    copy_paste=0.3,       # copy-paste augmentation (0..1)
    mosaic=1.0,           # mosaic (valor por defecto)
    cache=False           # cambiar a True solo si querés cachear imágenes
)


Ultralytics 8.3.203  Python-3.13.3 torch-2.8.0+cpu CPU (AMD Ryzen 5 PRO 3500U w/ Radeon Vega Mobile Gfx)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Silos\Base de datos\YOLO\processed_yolo_v2\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=silo_exp_aug_crops, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F8BDD25320>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [23]:
from ultralytics import YOLO
import glob, os

MODEL_PATH = r"runs\detect\silo_exp_aug_crops\weights\best.pt"  # revisar carpeta runs/ después de train
SRC = r"D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images"
OUT_DIR = r"runs\detect\predict_silo_v2"

model = YOLO(MODEL_PATH)
results = model.predict(source=SRC, imgsz=640, conf=0.1, save=True, save_txt=True, save_json=True)
# imprimir resumen
for r in results:
    print("Image:", r.path)
    if hasattr(r, 'boxes') and len(r.boxes) > 0:
        print("Boxes:", r.boxes.xyxy.cpu().numpy())
        print("Confs:", r.boxes.conf.cpu().numpy())
    else:
        print("No detecciones")



image 1/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_6_25_0_tile_r05_c05.jpg: 640x640 9 silos, 403.0ms
image 2/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_6_25_0_tile_r05_c05_C_256_0.jpg: 640x640 2 silos, 397.5ms
image 3/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_6_25_0_tile_r05_c05_C_256_1.jpg: 640x640 7 silos, 372.9ms
image 4/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_6_25_0_tile_r05_c05_C_512_0.jpg: 640x640 13 silos, 368.3ms
image 5/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_6_25_0_tile_r05_c05_C_512_1.jpg: 640x640 13 silos, 368.6ms
image 6/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_7_13_1_tile_r14_c09.jpg: 640x640 (no detections), 475.9ms
image 7/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020_7_13_1_tile_r14_c09_C_256_0.jpg: 640x640 (no detections), 389.7ms
image 8/32 D:\Silos\Base de datos\YOLO\processed_yolo_v2\test\images\2020

In [24]:
### prueba con claude